In [2]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator
load_dotenv()

True

In [3]:
model= ChatGoogleGenerativeAI(model="gemini-3-flash-preview",temperature=0.7)

class EvaluationSchema(BaseModel):
    """Schema for evaluation results."""
    score: float = Field(..., description="The evaluation score between 1 and 10.", ge=1.0, le=10.0)
    feedback: str = Field(..., description="Detailed feedback on the evaluation.")

structured_model=model.with_structured_output(EvaluationSchema)

In [4]:
class UpscState(TypedDict):
    """State for the UPSC preparation workflow."""
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_scores:Annotated[list[float],operator.add]
    avg_score:float

def evaluate_language(state:UpscState):
    prompt=f'Evaluate the language used in the following essay:\n\n{state["essay"]}\n\nProvide a score between 1 and 10 and detailed feedback.'
    output=structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score] }

def evalute_analysis(state:UpscState):
    prompt=f'Evaluate the analysis used in the following essay:\n\n{state["essay"]}\n\nProvide a score between 1 and 10 and detailed feedback.'
    output=structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score] }

def evaluate_clarity(state:UpscState):
    prompt=f'Evaluate the clarity of the following essay:\n\n{state["essay"]}\n\nProvide a score between 1 and 10 and detailed feedback.'
    output=structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score] }

def overall_evaluation(state:UpscState):
    prompt=f'Based on the following essay and the feedback provided on language, analysis, and clarity, provide an overall evaluation:\n\nEssay:\n{state["essay"]}\n\nLanguage Feedback:\n{state["language_feedback"]}\n\nAnalysis Feedback:\n{state["analysis_feedback"]}\n\nClarity Feedback:\n{state["clarity_feedback"]}\n\nProvide an overall score between 1 and 10 and detailed feedback.'
    output=model.invoke(prompt).content

    avg_score= sum(state['individual_scores'])/len(state['individual_scores'])
    return {'overall_feedback': output, 'avg_score': avg_score }

In [5]:
graph=StateGraph(UpscState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evalute_analysis)
graph.add_node('evaluate_clarity',evaluate_clarity)
graph.add_node('overall_evaluation',overall_evaluation)

graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_clarity')
graph.add_edge('evaluate_language','overall_evaluation')
graph.add_edge('evaluate_analysis','overall_evaluation')
graph.add_edge('evaluate_clarity','overall_evaluation')
graph.add_edge('overall_evaluation',END)

workflow=graph.compile()

In [6]:
initial_state={'essay': "The impact of climate change on global agriculture is profound and multifaceted. Rising temperatures, changing precipitation patterns, and increased frequency of extreme weather events are affecting crop yields and food security worldwide. In this essay, we will explore the various ways in which climate change is influencing agriculture and discuss potential adaptation strategies."}

workflow.invoke(initial_state)

{'essay': 'The impact of climate change on global agriculture is profound and multifaceted. Rising temperatures, changing precipitation patterns, and increased frequency of extreme weather events are affecting crop yields and food security worldwide. In this essay, we will explore the various ways in which climate change is influencing agriculture and discuss potential adaptation strategies.',
 'language_feedback': "The language used is formal, precise, and highly appropriate for an academic context. It employs sophisticated vocabulary such as 'multifaceted' and 'precipitation patterns' to convey complex ideas clearly. The structure of the introductory paragraph is logical, moving from a general statement to specific factors and then outlining the essay's objectives. The grammar and syntax are flawless, though the use of 'In this essay, we will...' is a somewhat conventional academic formula that, while correct, is standard for introductory writing.",
 'analysis_feedback': 'The provide